In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from helpers import make_preprocessor, evaluate


In [3]:
df = pd.read_csv('data/ames_housing.csv')
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.drop(index=[523, 1298])
X_train = train_df.drop(columns=['SalePrice', 'Id', 'GarageYrBlt', 'PoolArea'])
y_train = train_df['SalePrice']

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of y_train: {y_train.shape}")

Shape of X_train: (1166, 77)
Shape of y_train: (1166,)


In [5]:
from sklearn.linear_model import LinearRegression
mae, spread, rmse, r2 = evaluate(LinearRegression(), X_train, y_train)
print(mae, spread, rmse, r2)

17207.463617697722 612.8742141611775 25395.88625441448 0.8896753334786069


In [6]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest":     RandomForestRegressor(n_estimators=500, max_depth=20, max_features=0.3, min_samples_leaf=1, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42),
    "XGBoost":           XGBRegressor(n_estimators=1000, learning_rate=0.1, max_depth=2, random_state=42, n_jobs=-1),
    "LightGBM":          LGBMRegressor(n_estimators=600, learning_rate=0.1, num_leaves=4, random_state=42, n_jobs=-1, verbose=-1),
    "CatBoost":          CatBoostRegressor(random_state=42, verbose=0, allow_writing_files=False),
}

rows = []
for name, model in models.items():
    mae, spread, rmse, r2 = evaluate(model, X_train, y_train)
    rows.append({"Model": name, "MAE": mae, "Spread": spread, "RMSE": rmse, "R2": r2})

results = pd.DataFrame(rows).sort_values("MAE")
results.round({"MAE": 0, "Spread": 0, "RMSE": 0, "R2": 3})


,Model,MAE,Spread,RMSE,R2
5,CatBoost,14438.0,854.0,23436.0,0.908
4,LightGBM,14699.0,893.0,23377.0,0.908
3,XGBoost,14781.0,328.0,23093.0,0.910
2,Gradient Boosting,15227.0,780.0,23783.0,0.904
1,Random Forest,16583.0,1416.0,26331.0,0.883
0,Linear Regression,17207.0,613.0,25396.0,0.890


## All tuned models

Same cleaned training data, same preprocessing and same 5-fold split for every model. Each model uses the settings chosen in the tuning notebooks.

| Model | MAE | Spread (MAE) | RMSE | R² |
|---|---|---|---|---|
| CatBoost | $14,438 | 854 | $23,436 | 0.908 |
| LightGBM | $14,699 | 893 | $23,377 | 0.908 |
| XGBoost | $14,781 | 328 | $23,093 | 0.910 |
| Gradient Boosting | $15,227 | 780 | $23,783 | 0.904 |
| Random Forest | $16,583 | 1,416 | $26,331 | 0.883 |
| Linear Regression | $17,207 | 613 | $25,396 | 0.890 |

1. **CatBoost, LightGBM and XGBoost are effectively tied.** Their MAEs are within $343 of each other, less than their fold-to-fold spreads.
2. **XGBoost has the best RMSE and R² and the steadiest folds**, so it makes fewer large errors. CatBoost has the best MAE, so it is slightly better on typical houses.
3. **Random Forest beats the baseline on MAE but not on RMSE**, which means it makes some large errors.
4. **Caveat:** each tuned setting was chosen on the same folds it is scored on, so these numbers are slightly optimistic. The test set has not been used yet.

The final model will be chosen after trying stacking.